In [0]:
%pip install openpyxl
%restart_python


In [0]:
import pandas as pd 
import os
import pyspark
import openpyxl 
import datetime
import logging
import json
import requests
import urllib3
import time
import re
from datetime import datetime
from pathlib import Path
import configparser

def get_UserProfile_Reference():
    file_path = r"/Workspace/Users/baodi.wilkinson.external@atradius.com/SAP_BO_Analytics/BO Cluster Analysis/"
    # /Workspace/Users/baodi.wilkinson.external@atradius.com/SAP_BO_Analytics/BO Cluster Analysis/Arcade Risk Migration impact Databrick ingestion.xlsx
# /Workspace/Users/baodi.wilkinson.external@atradius.com
    try:
        # UserProfile_New_df = pd.read_excel(file_path + "User Properties Reference.xlsx", "User Details 10Sep2025")
        # UserProfile_Old_df = pd.read_excel(file_path + "User Properties Reference.xlsx", "User details 13Aug2025")
        # Audit_df = pd.read_excel(file_path + "Universe_Tables_IDT_UDT_W_Schema_Result.xlsx", "Universe_MetaData")
        # Audit_DF = pd.read_csv(file_path + "WEBI_Obsolete_records.csv")
        # Cluster_DF= pd.read_excel(file_path + "Control_Finance Team.xlsx", "Control_Finance_Flat")
        arcade_df = pd.read_excel(file_path + "Arcade Risk Migration impact Databrick ingestion.xlsx", "Master Sheet (2508205) - CoEx")
        # Cluster_DF = spark.createDataFrame(
        # print(UserProfile_New_df.head())
        # Tables_df = pd.read_csv(file_path + "doc_tables_full.csv")
    except FileNotFoundError:
        print("File not found")
        return None
    except Exception as e:
        print(f"Error: {e}")
        return None
    # return UserProfile_New_df, UserProfile_Old_df
    return arcade_df

def replace_Space_DF(source_Df: pyspark.sql.dataframe.DataFrame, replace_with:str="_"):
    logging.info("Replacing invalid characters in column names")
    # Replace all characters invalid for Delta: space , ; { } ( ) \n \t = * -
    new_columns = [re.sub(r'[ ,;{}()\n\t=*\-]+', replace_with, col).strip(replace_with) for col in source_Df.columns]
    return source_Df.toDF(*new_columns)

Todyay_Date = datetime.now().strftime("%Y-%m-%d")
# target_storage_account       = "dataplatform01ctrl01d"
# target_container_name      = "landing-zone"

target_storage_account       = "dataplatform01ext01d"
target_container_name      = "dataproduct-bomigrationdedev"
storage_account_location = (f"abfss://{target_container_name}@{target_storage_account}.dfs.core.windows.net")
#abfss://dataproduct-bomigrationdedev@dataplatform01ext01d.dfs.core.windows.net/
path = (f"api/bo_migration/sap_bo/load_date={Todyay_Date}/")
ingestion_path = (f"{storage_account_location}/{path}")
print(ingestion_path)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
# Todyay_Date = datetime.now().strftime("%Y-%m-%d")
logging.basicConfig(
    filename=f"Script_Dev_{Todyay_Date}.log",          # Dev Test Log file name
    # filename='Script_Prod_MB.log',          #Prod Log file name
    level=logging.INFO,          # Log level: DEBUG, INFO, WARNING, ERROR, CRITICAL
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logging.info(f"Script started: {Todyay_Date} with directory {os.getcwd()}")

# Store_User_DF=spark.createDataFrame(get_UserProfile_Reference()[0])
# Store_User_Old_DF=spark.createDataFrame(get_UserProfile_Reference()[1])
# Store_User_DF=replace_Space_DF(Store_User_DF)
# Store_User_Old_DF=replace_Space_DF(Store_User_Old_DF)
# print(Store_User_DF.head())
# print(Store_User_Old_DF.head())
# User_profile_new=(f"{ingestion_path}User_profile_new")
# User_profile_old=(f"{ingestion_path}User_profile_old")
# Store_User_DF.write.mode("overwrite").parquet(User_profile_new)
# Store_User_Old_DF.write.mode("overwrite").parquet(User_profile_old)
# Store_User_DF.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.bo_user_profile")
# Store_User_Old_DF.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.bo_user_profile_old")

# Audit_DF=spark.createDataFrame(get_UserProfile_Reference())
arcade_df=spark.createDataFrame(get_UserProfile_Reference())
from pyspark.sql.functions import lit, current_date

arcade_df = replace_Space_DF(arcade_df).withColumn("Ingestion_date", lit(datetime.now().strftime("%Y-%m-%d")))
# Audit_DF=replace_Space_DF(Audit_DF)
print(arcade_df.head())
# Audit_DF.write.mode("overwrite").parquet(f"{ingestion_path}Universe_WEBI_CMS_link")
# dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.bo_doc_sql_tables
arcade_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.arcaderisk_master_tables")
# dataplatform01_central_dev_catalog_01.bronze_sap_bo_snp.webi_full_records_metadata")

In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.arcaderisk_proj_impact

In [0]:
print(User_profile_new)
df1=spark.read.parquet(User_profile_new)
df1.printSchema()